# Mid Take Home Exam
## Web Data Scraping & Text Analysis

**Course:** 웹데이터수집과 텍스트분석(캡스톤디자인) (MGT4208-01)  
**Student:** 김대호  
**Student ID:** 20210673

## Data and Evaluation Rules

This report uses the **first 3,000 reviews** in `IMDB Dataset.csv`, in their original order. Positive reviews are encoded as **1** and negative reviews as **0**.

Question 1 evaluates dictionary-based sentiment predictions on all 3,000 reviews. Question 2 trains logistic regression models on the first 2,100 reviews and evaluates them on the remaining 900 reviews.

**Accuracy = number of correct predictions / number of evaluated reviews.**

All confusion matrices use **actual classes as rows** and **predicted classes as columns**, ordered as `[negative (0), positive (1)]`.

## 1. Dictionary-Based Sentiment Analysis

### 1.1. Answers

| Question | Quantity | Answer |
| :--- | :--- | ---: |
| (1)(a) | Number of positive reviews | **1,508** |
| (1)(b) | Percentage of negative reviews | **49.7333%** |
| (2)(a) | Mean AFINN score | **5.7466666667** |
| (2)(b) | AFINN prediction accuracy | **72.1000%** |
| (3)(a) | Mean VADER compound score | **0.2921087333** |
| (3)(b) | VADER prediction accuracy | **70.2333%** |
| (4)(a) | Mean TextBlob polarity score | **0.1016683911** |
| (4)(b) | TextBlob prediction accuracy | **68.8667%** |

There are 1,492 negative reviews. Their percentage is 1,492 / 3,000 × 100 = **49.7333%**.

### 1.2. Method

The dictionaries are applied to the **original review text**. The preprocessing specified in Question 2 is applied only in Question 2.

| Method | Score averaged | Rule for predicting positive (1) |
| :--- | :--- | :--- |
| AFINN | AFINN score | AFINN score > 0 |
| VADER | Compound score | Positive score > negative score |
| TextBlob | Polarity score | Polarity score > 0 |

All other cases, including ties, are classified as negative (0). VADER's compound score is used for the requested average; its positive and negative scores determine the predicted class.

### 1.3. Code and Execution Results

The following cells contain the complete implementation. The final cell executes the analysis and prints the means, confusion matrices, and classification reports.


**Imports and helper functions**

In [1]:
"""Q1: Dictionary-based sentiment analysis of the first 3,000 IMDB reviews."""
from pathlib import Path
import sys

BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if not (BASE / "IMDB Dataset.csv").exists():
    BASE = BASE / "midterm"
sys.path.insert(0, str(BASE / ".packages"))

import nltk
import pandas as pd
from afinn import Afinn
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from textblob import TextBlob

nltk.data.path.insert(0, str(BASE / "nltk_data"))


def evaluate(name, actual, predicted):
    accuracy = accuracy_score(actual, predicted)
    matrix = confusion_matrix(actual, predicted, labels=[0, 1])
    assert matrix.sum() == len(actual)
    assert matrix.trace() == (actual == predicted).sum()
    print(f"\n{name} accuracy: {accuracy:.10f} ({accuracy:.4%})")
    print("Confusion matrix: rows = actual, columns = predicted; order = [0, 1]")
    print(matrix)
    print(classification_report(actual, predicted, labels=[0, 1],
                                target_names=["negative (0)", "positive (1)"],
                                digits=6, zero_division=0))
    return accuracy

**Analysis procedure**

In [2]:
def main():
    reviews = pd.read_csv(BASE / "IMDB Dataset.csv", nrows=3000)
    assert len(reviews) == 3000 and reviews.notna().all().all()
    assert set(reviews["sentiment"]) == {"positive", "negative"}
    actual = reviews["sentiment"].map({"positive": 1, "negative": 0})
    print("Q1: First 3,000 reviews; positive = 1, negative = 0")
    print(f"(1)(a) Positive reviews: {actual.sum()}")
    print(f"(1)(b) Negative reviews: {(actual == 0).sum()}; percentage: {(actual == 0).mean():.4%}")

    # Q2's preprocessing is not applied to Q1. Preserve the original reviews.
    afinn = Afinn()
    vader = SentimentIntensityAnalyzer()
    scores = pd.DataFrame({"review_number": range(1, 3001), "actual": actual})
    scores["afinn"] = reviews["review"].map(afinn.score)
    vader_scores = pd.DataFrame(reviews["review"].map(vader.polarity_scores).tolist())
    scores["vader_compound"] = vader_scores["compound"]
    scores["vader_pos"] = vader_scores["pos"]
    scores["vader_neg"] = vader_scores["neg"]
    scores["textblob_polarity"] = reviews["review"].map(lambda text: TextBlob(text).sentiment.polarity)

    scores["afinn_prediction"] = (scores["afinn"] > 0).astype(int)
    # The exam requires pos > neg, NOT a compound-score threshold.
    scores["vader_prediction"] = (scores["vader_pos"] > scores["vader_neg"]).astype(int)
    scores["textblob_prediction"] = (scores["textblob_polarity"] > 0).astype(int)

    summary = []
    for number, name, score_column, prediction_column in [
        (2, "AFINN", "afinn", "afinn_prediction"),
        (3, "VADER", "vader_compound", "vader_prediction"),
        (4, "TextBlob", "textblob_polarity", "textblob_prediction"),
    ]:
        average = scores[score_column].mean()
        print(f"\n({number})(a) {name} mean score: {average:.10f}")
        accuracy = evaluate(f"({number})(b) {name}", actual, scores[prediction_column])
        summary.append({"method": name, "mean_score": average, "accuracy": accuracy})
    output = BASE / "results"
    output.mkdir(exist_ok=True)
    scores.to_csv(output / "q1_scores.csv", index=False)
    pd.DataFrame(summary).to_csv(output / "q1_summary.csv", index=False)
    return pd.DataFrame(summary)

**Run analysis and display results**

In [3]:
if __name__ == "__main__":
    q1_summary = main()

Q1: First 3,000 reviews; positive = 1, negative = 0
(1)(a) Positive reviews: 1508
(1)(b) Negative reviews: 1492; percentage: 49.7333%

(2)(a) AFINN mean score: 5.7466666667

(2)(b) AFINN accuracy: 0.7210000000 (72.1000%)
Confusion matrix: rows = actual, columns = predicted; order = [0, 1]
[[ 885  607]
 [ 230 1278]]
              precision    recall  f1-score   support

negative (0)   0.793722  0.593164  0.678941      1492
positive (1)   0.677984  0.847480  0.753316      1508

    accuracy                       0.721000      3000
   macro avg   0.735853  0.720322  0.716128      3000
weighted avg   0.735544  0.721000  0.716327      3000


(3)(a) VADER mean score: 0.2921087333

(3)(b) VADER accuracy: 0.7023333333 (70.2333%)
Confusion matrix: rows = actual, columns = predicted; order = [0, 1]
[[ 810  682]
 [ 211 1297]]
              precision    recall  f1-score   support

negative (0)   0.793340  0.542895  0.644648      1492
positive (1)   0.655382  0.860080  0.743906      1508

    accur

### 1.4. Interpretation

AFINN correctly classifies **2,163 of 3,000** reviews, VADER **2,107**, and TextBlob **2,066**. AFINN has the highest accuracy among these three methods under the classification rules specified in the exam.

## 2. Logistic Regression Using Review Terms

### 2.1. Answers

| Question | Input representation | Correct predictions | Test accuracy |
| :--- | :--- | ---: | ---: |
| (1) | Raw term frequency (`CountVectorizer`) | 748 / 900 | **83.1111%** |
| (2) | TF-IDF (`TfidfVectorizer`) | 756 / 900 | **84.0000%** |

### 2.2. Preprocessing and Model Settings

Each review is processed in the following order:

1. Remove HTML tags using BeautifulSoup with the `html5lib` parser.
2. Replace every character other than A–Z or a–z with a space.
3. Convert the text to lowercase and split it into tokens.
4. Remove NLTK English stopwords.
5. Join the remaining tokens with single spaces.

No stemming or lemmatization is added. The downloaded NLTK English stopword list contains 198 entries.

| Setting | Value |
| :--- | :--- |
| Training reviews | 1–2,100 |
| Test reviews | 2,101–3,000 |
| Shuffling | None |
| Vectorizer fitting | Training reviews only |
| Vectorizer parameters | Defaults, as shown in the exam hints |
| Classifier | `LogisticRegression()` with default parameters |
| Number of input features | 25,267 for each model |

The vocabulary and IDF weights are learned only from the training set. The test set is transformed using the fitted vectorizer. Classifier settings follow slide 34 of the sentiment analysis lecture materials.

### 2.3. Code and Execution Results


**Imports and helper functions**

In [4]:
"""Q2: Frequency/TF-IDF logistic regression, with an ordered 70/30 split."""
from pathlib import Path
import re
import sys
import warnings

BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if not (BASE / "IMDB Dataset.csv").exists():
    BASE = BASE / "midterm"
sys.path.insert(0, str(BASE / ".packages"))

import nltk
import pandas as pd
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.data.path.insert(0, str(BASE / "nltk_data"))


def preprocess(text, stop_words):
    # Input is review text, even when it happens to resemble a file path.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)
        text = BeautifulSoup(text, "html5lib").get_text()  # Step 1
    text = re.sub(r"[^a-zA-Z]", " ", text)             # Step 2
    tokens = text.lower().split()                       # Step 3
    tokens = [word for word in tokens if word not in stop_words]  # Step 4
    return " ".join(tokens)                             # Step 5

**Analysis procedure**

In [5]:
def main():
    reviews = pd.read_csv(BASE / "IMDB Dataset.csv", nrows=3000)
    assert len(reviews) == 3000 and reviews.notna().all().all()
    assert set(reviews["sentiment"]) == {"positive", "negative"}
    stop_words = set(stopwords.words("english"))
    clean = reviews["review"].map(lambda text: preprocess(text, stop_words))
    labels = reviews["sentiment"].map({"positive": 1, "negative": 0})
    x_train, x_test = clean.iloc[:2100], clean.iloc[2100:]
    y_train, y_test = labels.iloc[:2100], labels.iloc[2100:]
    assert len(x_train) == 2100 and len(x_test) == 900
    print("Q2: train = reviews 1-2100; test = reviews 2101-3000 (no shuffle)")
    print(f"NLTK English stopwords: {len(stop_words)}")
    print("Preprocessing complete: HTML, non-letters, and stopwords removed.")
    summary = []
    predictions = pd.DataFrame({"review_number": range(2101, 3001), "actual": y_test.to_numpy()})
    for name, vectorizer in [("CountVectorizer", CountVectorizer()),
                             ("TfidfVectorizer", TfidfVectorizer())]:
        # Fit the vocabulary and IDF only on the training reviews.
        vec = vectorizer.fit(x_train)
        train_matrix = vec.transform(x_train)
        test_matrix = vec.transform(x_test)
        model = LogisticRegression()  # Lecture slide 34: default settings.
        model.fit(train_matrix, y_train)
        predicted = model.predict(test_matrix)
        accuracy = accuracy_score(y_test, predicted)
        matrix = confusion_matrix(y_test, predicted, labels=[0, 1])
        assert matrix.sum() == 900 and matrix.trace() == (predicted == y_test).sum()
        print(f"\n{name} + LogisticRegression")
        print(f"Train shape: {train_matrix.shape}; test shape: {test_matrix.shape}")
        print(f"Iterations: {model.n_iter_.tolist()}; max_iter: {model.max_iter}")
        print(f"Test accuracy: {accuracy:.10f} ({accuracy:.4%}); correct: {matrix.trace()}/900")
        print("Confusion matrix: rows = actual, columns = predicted; order = [0, 1]")
        print(matrix)
        print(classification_report(y_test, predicted, labels=[0, 1],
                                    target_names=["negative (0)", "positive (1)"],
                                    digits=6, zero_division=0))
        summary.append({"method": name, "test_accuracy": accuracy,
                        "correct": int(matrix.trace()), "test_size": 900,
                        "features": train_matrix.shape[1], "iterations": int(model.n_iter_[0])})
        predictions[name] = predicted
    output = BASE / "results"
    output.mkdir(exist_ok=True)
    predictions.to_csv(output / "q2_predictions.csv", index=False)
    pd.DataFrame(summary).to_csv(output / "q2_summary.csv", index=False)
    return pd.DataFrame(summary)

**Run analysis and display results**

In [6]:
if __name__ == "__main__":
    q2_summary = main()

Q2: train = reviews 1-2100; test = reviews 2101-3000 (no shuffle)
NLTK English stopwords: 198
Preprocessing complete: HTML, non-letters, and stopwords removed.

CountVectorizer + LogisticRegression
Train shape: (2100, 25267); test shape: (900, 25267)
Iterations: [37]; max_iter: 100
Test accuracy: 0.8311111111 (83.1111%); correct: 748/900
Confusion matrix: rows = actual, columns = predicted; order = [0, 1]
[[355  93]
 [ 59 393]]
              precision    recall  f1-score   support

negative (0)   0.857488  0.792411  0.823666       448
positive (1)   0.808642  0.869469  0.837953       452

    accuracy                       0.831111       900
   macro avg   0.833065  0.830940  0.830809       900
weighted avg   0.832956  0.831111  0.830841       900


TfidfVectorizer + LogisticRegression
Train shape: (2100, 25267); test shape: (900, 25267)
Iterations: [24]; max_iter: 100
Test accuracy: 0.8400000000 (84.0000%); correct: 756/900
Confusion matrix: rows = actual, columns = predicted; order =

### 2.4. Interpretation

The TF-IDF model correctly classifies **eight more test reviews** than the raw-frequency model. Its accuracy is higher by **0.8889 percentage points**. Both models converge within the default limit of 100 iterations: 37 iterations for raw frequencies and 24 for TF-IDF.

Question 1 and Question 2 use different evaluation sets: all 3,000 reviews in Question 1 and the 900 held-out reviews in Question 2. Their accuracies therefore describe different evaluation procedures.

## 3. Reproducibility

Place `IMDB Dataset.csv` in the same folder as this notebook. Install the packages listed below, then download the NLTK resources once:

```python
import nltk
nltk.download('stopwords')
nltk.download('vader_lexicon')
```

Run the notebook cells from top to bottom. The analysis also saves review-level scores and predictions as CSV files in `results/`. The recorded package versions used for this report are shown below.


In [7]:
import sys
from pathlib import Path
from importlib.metadata import version
base = Path.cwd()
sys.path.insert(0, str(base / '.packages'))
print('Python:', sys.version)
for package in ['pandas', 'scikit-learn', 'nltk', 'afinn', 'textblob', 'beautifulsoup4', 'html5lib']:
    print(package, version(package))


Python: 3.10.21 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:35:39) [MSC v.1942 64 bit (AMD64)]
pandas 2.2.3
scikit-learn 1.6.1
nltk 3.9.1
afinn 0.1
textblob 0.20.1
beautifulsoup4 4.12.3
html5lib 1.1


## 4. Numerical Results Summary

The tables below retain the numerical results returned by the two analysis procedures.

In [8]:
from IPython.display import display
display(q1_summary)
display(q2_summary)


,method,mean_score,accuracy
0,AFINN,5.746667,0.721000
1,VADER,0.292109,0.702333
2,TextBlob,0.101668,0.688667


,method,test_accuracy,correct,test_size,features,iterations
0,CountVectorizer,0.831111,748,900,25267,37
1,TfidfVectorizer,0.840000,756,900,25267,24
